# MLPによる手書き数字認識と表現学習 / Handwritten digits and representation learning
**CNNは使いません。** 8×8の画像の行を順番に連結して64次元ベクトルにし、通常の全結合MLPで学習します。

**手順：全セル実行 → Start → Test once → 下段の Analyze features**。
1. 画素から数字0〜9を認識することを学習。
2. 独立テストデータで正解率と混同行列を確認。
3. 隠れ層を選び、元の画素・未学習モデル・学習後モデルを比較。
4. 特徴空間のPCA表示、固定特徴を使う分類、類似画像検索を観察。

CPUで動作。コメント日英併記。使用するのはscikit-learn付属Digits（1797枚、8×8）で、MNISTではありません。外部ダウンロード不要です。

In [ ]:
# ライブラリ / Libraries
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import traceback
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
# 層ごとの行列演算はNumPy / NumPy matrix operations for every layer


## 画像をベクトルにする / Flattening images
画素値0〜16を16で割り、0〜1に正規化。64個の明るさを入力します。位置の並びは固定ですが、画像の近傍関係を利用する特別な構造はありません。

学習60%・検証20%・テスト20%に層化分割。学習中に見るのは検証データです。**Test once**は学習後の最終評価用で、設定選択には使わないでください。

最終層の10個のスコアをSoftmaxで確率に変換し、多クラス交差エントロピーを最小化します。Adam・He初期化・ミニバッチを使用。

In [ ]:
def softmax(z):
    # 最大値を引き、オーバーフローを防ぐ / Shift for numerical stability
    e=np.exp(z-z.max(axis=1,keepdims=True))
    return e/e.sum(axis=1,keepdims=True)

class DeepMLP:
    def __init__(self,depth=3,width=64,seed=4):
        rng=np.random.default_rng(seed); sizes=[64]+[width]*depth+[10]
        self.W=[]; self.b=[]
        for layer,(a,b) in enumerate(zip(sizes[:-1],sizes[1:])):
            # 隠れ層はHe、出力層は控えめな初期値 / He initialization for hidden layers
            scale=np.sqrt(2/a) if layer<depth else np.sqrt(1/a)
            self.W.append(rng.normal(0,scale,(a,b))); self.b.append(np.zeros((1,b)))
        self.m=[np.zeros_like(p) for p in self.params()]; self.v=[np.zeros_like(p) for p in self.params()]; self.step=0
    def params(self):
        return [p for pair in zip(self.W,self.b) for p in pair]
    def forward(self,X):
        A=[X]; Z=[]
        for i,(W,b) in enumerate(zip(self.W,self.b)):
            z=A[-1]@W+b; Z.append(z)
            A.append(np.maximum(z,0) if i<len(self.W)-1 else softmax(z))
        return A,Z
    def loss_grad(self,X,t):
        A,Z=self.forward(X); shift=Z[-1]-Z[-1].max(axis=1,keepdims=True); loss=np.mean(np.log(np.exp(shift).sum(axis=1))-shift[np.arange(len(X)),t]); delta=A[-1].copy(); delta[np.arange(len(X)),t]-=1; delta/=len(X)
        gradients=[None]*(2*len(self.W))
        for i in reversed(range(len(self.W))):
            gradients[2*i]=A[i].T@delta; gradients[2*i+1]=delta.sum(axis=0,keepdims=True)
            if i: delta=(delta@self.W[i].T)*(Z[i-1]>0)
        return float(loss),gradients,A
    def update(self,X,t,lr=.003):
        loss,G,A=self.loss_grad(X,t); self.step+=1
        for i,(p,g) in enumerate(zip(self.params(),G)):
            self.m[i]=.9*self.m[i]+.1*g; self.v[i]=.999*self.v[i]+.001*g*g
            p-=lr*(self.m[i]/(1-.9**self.step))/(np.sqrt(self.v[i]/(1-.999**self.step))+1e-8)
        return loss
    def metrics(self,X,t):
        A,Z=self.forward(X)
        shift=Z[-1]-Z[-1].max(axis=1,keepdims=True)
        return float(np.mean(np.log(np.exp(shift).sum(axis=1))-shift[np.arange(len(X)),t])),float(np.mean(A[-1].argmax(axis=1)==t))


## 操作 / Controls
**Reset → Start**。設定を変えたらReset。学習中に中断する場合はColabの実行停止（■）を使用。Startで続きから再開できます。

上段：検証画像12枚の正解T・予測P（赤枠=誤分類）。下段：損失・精度・選択した画像の10クラス確率。画像の組は学習中も固定です。

Epochは学習データ全体を1回使用する単位。ネットワークの深さ・幅を変更して比較してください。深いほど必ず良いとは限りません。

In [ ]:
depth=widgets.IntSlider(value=3,min=1,max=6,description='Hidden layers')
width=widgets.Dropdown(options=[16,32,64,128],value=64,description='Width')
epochs=widgets.IntSlider(value=60,min=10,max=150,step=10,description='Epochs')
rate=widgets.FloatText(value=.001,description='Rate')
seed=widgets.IntSlider(value=4,min=0,max=20,description='Init seed')
select=widgets.IntSlider(value=0,min=0,max=11,description='Image',continuous_update=False)
reset=widgets.Button(description='Reset',button_style='info'); start=widgets.Button(description='Start / Resume',button_style='success'); step=widgets.Button(description='Step +5 epochs'); test=widgets.Button(description='Test once',button_style='warning')
status=widgets.HTML(); out=widgets.Output(); testout=widgets.Output(); state={}
controls=[depth,width,epochs,rate,seed,select,reset,start,step,test]

def record():
    m=state['model']; a=m.metrics(state['X'],state['t']); b=m.metrics(state['Xv'],state['tv'])
    state['history'].append([state['epoch'],*a,*b])

def render():
    if not state: return
    m=state['model']; Xv,tv=state['Xv'],state['tv']; probs=m.forward(Xv[:12])[0][-1]; pred=probs.argmax(axis=1); hist=np.array(state['history'])
    with out:
        clear_output(wait=True)
        fig=plt.figure(figsize=(15,8)); gs=fig.add_gridspec(3,6,height_ratios=[1,1,2])
        for i in range(12):
            ax=fig.add_subplot(gs[i//6,i%6]); ax.imshow(Xv[i].reshape(8,8),cmap='gray_r',vmin=0,vmax=1); correct=pred[i]==tv[i]
            ax.set_title(f'{i}: T={tv[i]}  P={pred[i]}',color='#168070' if correct else '#d63b30',fontsize=11); ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_edgecolor('#168070' if correct else '#d63b30'); sp.set_linewidth(2)
        ax=fig.add_subplot(gs[2,:2]); ax.plot(hist[:,0],hist[:,1],label='Train'); ax.plot(hist[:,0],hist[:,3],label='Validation'); ax.set(xlabel='Epoch',ylabel='Cross-entropy',title='Loss'); ax.legend(); ax.grid(alpha=.2)
        ax=fig.add_subplot(gs[2,2:4]); ax.plot(hist[:,0],hist[:,2],label='Train'); ax.plot(hist[:,0],hist[:,4],label='Validation'); ax.set(xlabel='Epoch',ylabel='Accuracy',ylim=(0,1.02),title='Accuracy'); ax.legend(); ax.grid(alpha=.2)
        ax=fig.add_subplot(gs[2,4:]); k=select.value; ax.bar(np.arange(10),probs[k],color=['#ed7d31' if j==tv[k] else '#38899e' for j in range(10)]); ax.set(xticks=np.arange(10),ylim=(0,1),title=f'Image {k}: class probabilities',xlabel='Digit',ylabel='Softmax output')
        fig.suptitle('64 pixels → '+' → '.join([str(state['width'])]*state['depth'])+' → 10 classes | epoch '+str(state['epoch']),fontsize=15)
        fig.tight_layout(); display(fig); plt.close(fig)
    status.value=f'<b>Epoch {state["epoch"]}/{state["limit"]}</b> | train {hist[-1,2]:.1%} | validation {hist[-1,4]:.1%} | '+('Running' if state['running'] else 'Ready / Paused')

def initialize(_=None):
    if state.get('running'): return
    if not np.isfinite(rate.value) or rate.value<=0: status.value='Rate must be positive and finite.'; return
    d=load_digits(); X=d.data/16.; t=d.target
    # 分割は固定し、初期重みだけを変更 / Fixed split; initialization seed is independent
    Xa,Xt,ta,tt=train_test_split(X,t,test_size=.2,random_state=42,stratify=t)
    X,Xv,t,tv=train_test_split(Xa,ta,test_size=.25,random_state=42,stratify=ta)
    state.clear(); state.update(model=DeepMLP(depth.value,width.value,seed.value),initial_model=DeepMLP(depth.value,width.value,seed.value),X=X,t=t,Xv=Xv,tv=tv,Xt=Xt,tt=tt,history=[],epoch=0,limit=epochs.value,rate=rate.value,rng=np.random.default_rng(seed.value+100),width=width.value,depth=depth.value,running=False,tested=False)
    with testout: clear_output()
    record(); render()
    if 'sync_analysis_controls' in globals(): sync_analysis_controls()

def train_epochs(count=None):
    if state.get('running'): return
    # テスト後の学習は新しい実験としてReset / Reset after final testing
    if state['tested']: status.value='Test already evaluated. Reset for a new experiment.'; return
    state['running']=True
    for w in controls: w.disabled=True
    error=None
    try:
        end=state['limit'] if count is None else min(state['limit'],state['epoch']+count)
        while state['epoch']<end:
            idx=state['rng'].permutation(len(state['X']))
            for a in range(0,len(idx),64):
                batch=idx[a:a+64]; state['model'].update(state['X'][batch],state['t'][batch],state['rate'])
            state['epoch']+=1
            state['analysis']=None
            if not all(np.isfinite(p).all() for p in state['model'].params()): raise FloatingPointError('Lower Rate and Reset.')
            record()
            if state['epoch']%2==0 or state['epoch']==end: render()
    except KeyboardInterrupt: error='Interrupted. Start resumes.'
    except Exception: error=traceback.format_exc()
    finally:
        state['running']=False
        for w in controls: w.disabled=False
        render()
        if error:
            with out: print(error)

def final_test(_=None):
    if state.get('running') or state.get('tested'): return
    if not state['epoch']: status.value='Train the model before Test once.'; return
    state['tested']=True; m=state['model']; loss,acc=m.metrics(state['Xt'],state['tt']); pred=m.forward(state['Xt'])[0][-1].argmax(axis=1)
    with testout:
        clear_output(wait=True); print(f'Final test accuracy: {acc:.2%} | loss: {loss:.5f} | N={len(pred)}')
        cm=confusion_matrix(state['tt'],pred,labels=np.arange(10)); fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(cm,cmap='Blues'); fig.colorbar(im,ax=ax)
        for i in range(10):
            for j in range(10): ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black',fontsize=8)
        ax.set(xticks=np.arange(10),yticks=np.arange(10),xlabel='Predicted digit',ylabel='True digit',title='Final test confusion matrix'); fig.tight_layout(); display(fig); plt.close(fig)
reset.on_click(initialize); start.on_click(lambda _:train_epochs()); step.on_click(lambda _:train_epochs(5)); test.on_click(final_test)
select.observe(lambda _:render() if not state.get('running') else None,names='value')
display(widgets.VBox([widgets.HBox([depth,width,epochs]),widgets.HBox([rate,seed]),widgets.HBox([reset,start,step,test]),status,select,out,testout]))
initialize()


### ボタンを使わない代替 / Direct execution fallback
ボタンが反応しない場合だけ下のFalseをTrueに変更し、セルを実行します。

In [ ]:
RUN_WITHOUT_BUTTON = False
if RUN_WITHOUT_BUTTON:
    train_epochs()
# 最終テスト / Final test: run final_test() after choosing the final model.


## 観察 / Observations
- 最初は誤分類が多くても、学習によって予測と確率が変化します。
- 検証損失が上がるのに学習損失が下がる場合、過学習の兆候です。
- テスト結果を見て設定を選び直すと、厳密な独立評価ではなくなります。
- 全結合NNでも数字を分類できますが、画像の位置ずれなどへの強さを自動的に保証するものではありません。

Data: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html
This is 8×8 Digits, not MNIST. No convolution is used. All weights and gradients are computed with NumPy.

## 2. 隠れ層の表現を確かめる / Inspect learned representations
**StartまたはStepで学習した後**に実行します。Test onceは解析の必須条件ではありません。層番号は1始まりです。実際に存在する層だけを選べます。Digitは数字の種類（0〜9）、Exampleはその数字の画像番号です。元の画素、同じ初期値の未学習MLP、学習後MLPの3種類を比較します。

- **PCA**：各表現について学習データだけでPCAを求め、検証データを2次元表示。色は正解ラベルで、PCA計算にはラベルを使いません。各図は異なる座標系です。2次元の見た目だけで分類能力は判断できません。
- **固定特徴での分類（linear probe）**：特徴抽出器を更新せず、学習特徴上で新しいロジスティック回帰を学習し、検証データで評価。
- **類似検索**：検証画像を問い合わせ、学習画像から近い画像を検索。正解ラベルは検索に使いません。

距離はL2正規化後のユークリッド距離（コサイン類似度と対応、ゼロベクトルを除く）で統一します。linear probeにも同じ正規化特徴を使用します。テストセットは解析中に使いません。解析結果を見た設定変更は探索的実験です。

PCA is a projection, not the network's actual feature dimension. A random network may already separate some samples. Learned features need not outperform pixels on every metric. Feature coordinates have no guaranteed human-readable meaning.

In [ ]:
# 隠れ層の特徴抽出と比較 / Extract and compare hidden representations
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import normalize
import pandas as pd

layer=widgets.Dropdown(options=list(range(1,len(state['model'].W))),value=len(state['model'].W)-1,description='Hidden layer')
digit=widgets.Dropdown(options=list(range(10)),value=0,description='Digit')
query=widgets.IntSlider(value=0,min=0,max=1,description='Example',continuous_update=False)
analyze=widgets.Button(description='Analyze features',button_style='primary')
search=widgets.Button(description='Find similar images')
rep_status=widgets.HTML(); rep_out=widgets.Output(); search_out=widgets.Output()

def representations(layer_number):
    if layer_number>len(state['model'].W)-1:
        raise ValueError('Choose a layer from 1 to '+str(len(state['model'].W)-1))
    k=layer_number
    current=state['model']; initial=state['initial_model']
    return [
        ('Pixels',state['X'],state['Xv']),
        ('Untrained hidden',initial.forward(state['X'])[0][k],initial.forward(state['Xv'])[0][k]),
        ('Trained hidden',current.forward(state['X'])[0][k],current.forward(state['Xv'])[0][k])]

def analyze_features(_=None):
    if state.get('running'): rep_status.value='Wait until training finishes.'; return
    if not state.get('epoch'): rep_status.value='まずStartまたはStepで学習してください / Train first.'; return
    analyze.disabled=True
    rep_status.value='解析中 / Computing features and probes…'
    try:
        reps=representations(layer.value); results=[]; cache=[]
        with rep_out:
            clear_output(wait=True)
            fig,axes=plt.subplots(1,3,figsize=(15,4.4))
            for ax,(name,F,Fv) in zip(axes,reps):
                # 同じ距離定義で比較 / Use the same normalization and distance
                Fn=normalize(F); Fvn=normalize(Fv)
                pca=PCA(n_components=2).fit(Fn); coords=pca.transform(Fvn)
                ax.scatter(coords[:,0],coords[:,1],c=state['tv'],cmap='tab10',vmin=-.5,vmax=9.5,s=18,alpha=.8)
                ax.set(title=f'{name}: {F.shape[1]}D → 2D',xlabel='PC1',ylabel='PC2')
                # 隠れ層は固定、分類器だけを学習 / Freeze features, train only a new classifier
                probe=LogisticRegression(C=10,max_iter=2000).fit(Fn,state['t'])
                nearest=KNeighborsClassifier(n_neighbors=1).fit(Fn,state['t'])
                results.append({'Representation':name,'Dimensions':F.shape[1],
                    'Linear probe validation accuracy':probe.score(Fvn,state['tv']),
                    '1-NN validation accuracy':nearest.score(Fvn,state['tv']),
                    'Zero-vector fraction':float(np.mean(np.linalg.norm(Fv,axis=1)==0))})
                cache.append((name,Fn,Fvn))
            from matplotlib.lines import Line2D
            handles=[Line2D([0],[0],marker='o',linestyle='',color=plt.cm.tab10(d/9),label=str(d)) for d in range(10)]
            fig.legend(handles=handles,loc='lower center',ncol=10); fig.tight_layout(rect=[0,.07,1,1]); display(fig); plt.close(fig)
            table=pd.DataFrame(results)
            for col in ['Linear probe validation accuracy','1-NN validation accuracy','Zero-vector fraction']:
                table[col]=table[col].map(lambda value:f'{value:.1%}')
            display(table)
        state['analysis']={'epoch':state['epoch'],'layer':layer.value,'cache':cache}
        rep_status.value=f'Layer {layer.value}, epoch {state["epoch"]}. PCA and probes fitted on training data only.'
        find_similar()
    except Exception:
        rep_status.value='解析エラー / Analysis error: see details below.'
        with rep_out: print(traceback.format_exc())
    finally: analyze.disabled=False

def find_similar(_=None):
    info=state.get('analysis')
    if state.get('running'): return
    if not info or info['epoch']!=state['epoch'] or info['layer']!=layer.value:
        analyze_features()
        return
    candidates=np.flatnonzero(state['tv']==digit.value)
    k=int(candidates[min(query.value,len(candidates)-1)])
    with search_out:
        clear_output(wait=True)
        fig,axes=plt.subplots(3,7,figsize=(12,6))
        for row,(name,F,Fv) in enumerate(info['cache']):
            # ラベルを使わず学習画像を距離で並べる / Retrieve training images without using labels
            dist=np.sum((F-Fv[k])**2,axis=1); ids=np.argsort(dist)[:6]
            axes[row,0].imshow(state['Xv'][k].reshape(8,8),cmap='gray_r',vmin=0,vmax=1)
            axes[row,0].set_title(f'{name}\nQuery T={state["tv"][k]}',fontsize=10)
            for col,idx in enumerate(ids,1):
                axes[row,col].imshow(state['X'][idx].reshape(8,8),cmap='gray_r',vmin=0,vmax=1)
                axes[row,col].set_title(f'T={state["t"][idx]} | d={np.sqrt(dist[idx]):.2f}',fontsize=9)
            for ax in axes[row]: ax.set_xticks([]); ax.set_yticks([])
        fig.suptitle('Query: validation image | Neighbors: training images',fontsize=13); fig.tight_layout(); display(fig); plt.close(fig)
def choose_digit(change=None):
    # 数字ラベルと画像の番号を区別 / Separate class label from sample index
    candidates=np.flatnonzero(state['tv']==digit.value)
    query.value=0; query.max=max(0,len(candidates)-1)
    if state.get('analysis'): find_similar()

def choose_layer(change=None):
    state['analysis']=None
    rep_status.value='層を変更しました。Analyze featuresを押してください / Layer changed: click Analyze features.'

def sync_analysis_controls():
    layer.options=list(range(1,len(state['model'].W)))
    layer.value=len(state['model'].W)-1
    choose_digit()
    with rep_out: clear_output()
    with search_out: clear_output()
    rep_status.value='学習後にAnalyze featuresを押してください / Train, then Analyze features.'

analyze.on_click(analyze_features); search.on_click(find_similar)
digit.observe(choose_digit,names='value')
query.observe(lambda change:find_similar() if state.get('analysis') else None,names='value')
layer.observe(choose_layer,names='value')
choose_digit()
display(widgets.VBox([widgets.HBox([layer,analyze]),rep_status,rep_out,widgets.HBox([digit,query,search]),search_out]))


### 観察ポイント / Questions to explore
1. 学習後に、同じ数字の検証画像がまとまるようになったでしょうか。
2. 元の画素との比較で、linear probeや1-NNの精度はどう変わりましたか。
3. 前の層と後の層で、検索される画像はどう変わりますか。
4. 誤分類された画像は、特徴空間でも別の数字に近いでしょうか。

特徴抽出器を固定したまま、新しい分類器だけを学習できることがポイントです。End-to-Endで獲得した表現を、別の計算に再利用しています。

### 後半が反応しない場合の直接実行 / Direct analysis fallback
下のセルはボタンを使わず解析を実行します。先にStartまたはStepで学習してください。`DIGIT_TO_INSPECT` は0〜9、`LAYER_TO_INSPECT` は1〜隠れ層数に変更できます。

In [ ]:
# FalseをTrueにしてこのセルを実行 / Set True and run this cell
RUN_FEATURE_ANALYSIS = False
DIGIT_TO_INSPECT = 0
LAYER_TO_INSPECT = 3
if RUN_FEATURE_ANALYSIS:
    if not 0 <= DIGIT_TO_INSPECT <= 9:
        raise ValueError('Digit must be 0–9.')
    if not 1 <= LAYER_TO_INSPECT <= len(state['model'].W)-1:
        raise ValueError('Choose an existing hidden layer.')
    layer.value=LAYER_TO_INSPECT
    digit.value=DIGIT_TO_INSPECT
    analyze_features()
